# Choosing a Voice

The narrator's voice is the most personal part of an audiobook — the right choice makes a book easy to listen to for hours; the wrong one creates constant low-level friction. This notebook walks through vorpal's full voice suite so you can make an informed choice before committing to a full synthesis run.

vorpal ships with 11 voices: 8 single Kokoro voices and 3 blend voices. All of them are local — no cloud calls, no account, no per-character cost. The blend voices are particularly interesting: they're weighted combinations of two single voices, producing a distinct character without any additional training.

We'll also cover the `vorpal voices --sample` command, which synthesizes a short passage in every available voice so you can hear them before deciding.

In [ ]:
import subprocess
import json

from vorpal.tts.voices import VOICE_REGISTRY

# Try to display as a table; fall back to plain print if pandas isn't available
try:
    import pandas as pd
    rows = []
    for name, spec in VOICE_REGISTRY.items():
        kind = "blend" if spec.get("blend") else "single"
        description = spec.get("description", "")
        rows.append({"name": name, "kind": kind, "description": description})
    df = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", 80)
    display(df)
except ImportError:
    print(f"{'NAME':<30} {'KIND':<8} DESCRIPTION")
    print("-" * 90)
    for name, spec in VOICE_REGISTRY.items():
        kind = "blend" if spec.get("blend") else "single"
        desc = spec.get("description", "")
        print(f"{name:<30} {kind:<8} {desc}")

## Single Voices

The single voices are drawn directly from the Kokoro 82M model's built-in speaker embeddings. Each one is a distinct trained voice with its own timbre, cadence, and accent.

The naming convention is: `{accent}_{gender}_{name}` — so `af_heart` is American female "Heart", `bm_george` is British male "George", and so on.

**American voices:**
- `af_heart` — warm, clear, slightly intimate; the default narrator. Works well for most genres.
- `af_nova` — crisper and more neutral than Heart; good for non-fiction and instructional material.
- `af_sky` — lighter and slightly higher; suits lighter fiction and young adult.
- `am_echo` — American male, warm mid-range; good all-purpose male narrator.
- `am_michael` — American male, steady and measured; well-suited to long-form non-fiction.
- `am_fenrir` — deep, commanding American male; strongest presence in the suite.

**British voices:**
- `bf_emma` — British female, clear RP accent; suits British literature and period fiction.
- `bm_george` — distinguished British male, measured delivery; excellent for Victorian and Edwardian material.

In [ ]:
# Print single voices grouped by accent
single_voices = {name: spec for name, spec in VOICE_REGISTRY.items() if not spec.get("blend")}

american = {n: s for n, s in single_voices.items() if n.startswith(("af_", "am_"))}
british  = {n: s for n, s in single_voices.items() if n.startswith(("bf_", "bm_"))}

print("AMERICAN VOICES")
print("-" * 60)
for name, spec in american.items():
    print(f"  {name:<20} {spec.get('description', '')}")

print()
print("BRITISH VOICES")
print("-" * 60)
for name, spec in british.items():
    print(f"  {name:<20} {spec.get('description', '')}")

## Blend Voices

Blend voices are a simple but effective technique: instead of choosing one speaker embedding, vorpal computes a weighted average of two embeddings at inference time. The result is a voice that sits between its two parents — inheriting qualities from each without being either one.

No training is required. The blend is specified as a dict of `{voice_name: weight}` in the voice registry, and the blend recipe is included in the TTS cache key — so `blend_warm_bright` at 65/35 and at 70/30 are cached separately.

The three current blends:

- **`blend_warm_bright`** (Heart 65% + Nova 35%) — takes Heart's warmth and adds Nova's crispness. Good general-purpose narrator that's slightly more precise than Heart alone.
- **`blend_deep_steady`** (Fenrir 55% + Michael 45%) — Fenrir's authority anchored by Michael's steadiness. Used for the Trotsky military writings production. Best for serious non-fiction, history, political or military material.
- **`blend_transatlantic`** (Heart 50% + Emma 50%) — equal mix of American and British female. Produces a mid-Atlantic quality that works well for early 20th century literature and translations.

In [ ]:
# Show the blend recipes from the voice registry
blend_voices = {name: spec for name, spec in VOICE_REGISTRY.items() if spec.get("blend")}

print("BLEND VOICES")
print("-" * 70)
for name, spec in blend_voices.items():
    desc = spec.get("description", "")
    recipe = spec["blend"]
    recipe_str = " + ".join(f"{v} {w*100:.0f}%" for v, w in recipe.items())
    print(f"  {name}")
    print(f"    Description: {desc}")
    print(f"    Recipe:      {recipe_str}")
    print()

## Auditioning Voices

Reading descriptions is no substitute for listening. `vorpal voices --sample` synthesizes a short passage in every available voice and writes one WAV file per voice to a `voice_samples/` directory.

You can pass any text you like with `--text`. Using a passage from your actual book is more useful than a generic test sentence — you'll hear how the voice handles the book's specific vocabulary, sentence length, and rhythm.

The sample below uses a line from Trotsky's military writings. Swap it for a sentence from your book.

In [ ]:
sample_text = "The army is not merely a fighting force; it is also a school. What is it teaching, and to whom?"

result = subprocess.run(
    ["vorpal", "voices", "--sample", "--text", sample_text],
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else result.stderr)

## Choosing a Voice for Your Book

There are no hard rules, but here is the guidance that has held up through production use:

**Political, military, or historical non-fiction** — `blend_deep_steady`. The Fenrir/Michael blend has the authority to carry long stretches of analytical prose without tiring the listener. This is what was used for the 19-hour Trotsky production.

**Literary fiction, contemporary** — `af_heart` (default) or `blend_warm_bright`. Heart is the most versatile voice in the suite; the blend version adds a little more precision without losing warmth.

**British classics and period fiction** — `bm_george` for a dignified male narrator; `bf_emma` for female-voiced narration. For a mid-Atlantic option that doesn't commit fully to either accent, try `blend_transatlantic`.

**Science fiction and speculative fiction** — `am_echo` or `af_nova`. Both have a slightly more neutral, precise delivery that suits speculative content without making it sound clinical.

**Theatrical plays** — vorpal casts these automatically. In play mode, the voice flag you pass becomes the default narrator voice, and characters are assigned voices from the rest of the suite based on gender and type.

**When unsure** — run `--sample` with a paragraph from your book and trust your ear over any recommendation.

## Using a Voice in a Build

Pass `--voice <name>` to `vorpal build`. The voice name must exactly match an entry in the registry (case-sensitive).

```bash
vorpal build book.epub --voice blend_deep_steady --output book_deep
vorpal build book.epub --voice bm_george --output book_george
vorpal build book.epub --voice af_heart  # default; same as omitting --voice
```

The voice choice is recorded in `book.json` under `settings.voice`. If you rebuild the same workdir with a different voice, only the synthesis stage re-runs — extraction and chapter detection are skipped. Chunks that were synthesized with the previous voice are not deleted; they stay in the cache under their old key. Storage is the only cost.

In [ ]:
# Construct a build command with voice selection — print it rather than run it
# so you can review before committing to a full synthesis.

voice = "blend_deep_steady"  # change this to your chosen voice
input_file = "book.epub"      # change this to your input file
output_stem = "book_deep"     # change this to your desired output name

cmd = ["vorpal", "build", input_file, "--voice", voice, "--output", output_stem]
print("Command to run:")
print(" ".join(cmd))
print()
print("Add --end-page 30 for a quick test before building the full book.")

## What's Coming

The local Kokoro voice suite covers a wide range of narration styles, but it has one limitation: it doesn't do expressive delivery. Every passage is read in the same measured tone regardless of whether it's a tense confrontation or a quiet description.

Two upcoming features address this:

**Tone detection (Phase 8)** — a fast LLM pass (Claude Haiku) tags each paragraph with a tone label (`neutral`, `somber`, `tense`, `warm`, `wry`, `urgent`, `reflective`). These tags feed into TTS engines that support style variation. The local Kokoro engine will use them for subtle pitch and rate adjustments; cloud engines will use them for full style switching.

**API TTS engines (Phase 7)** — `--engine openai` (gpt-4o-mini-tts, ~$5–10/book), `--engine azure` (SSML styles), `--engine elevenlabs` (best voice quality available, ~$50–150/book). These are opt-in — the local engine remains the default. See `docs/upcoming.md` for the full picture of what's coming and when.

Custom voice training (Phase 9) is also on the roadmap as an in-house spike — the goal is a small suite of voices that are distinctly vorpal's own, trained on properly licensed data. Voice cloning (copying a real person's voice) is explicitly not on the roadmap and never will be.